# Team model experiment template
Copy this notebook for one model/config. Fill in the owner, hypothesis, and config path. All teammates use the same audited dataset and frozen validation split. The shared trainer and configs are implementation tasks; this notebook does not claim training is ready yet.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'plan.md').exists():
    raise RuntimeError('Start Jupyter from the repository root, or change directory to the repository before running this notebook.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.paths import get_data_dir
DATA_DIR = get_data_dir()
print('Dataset cache:', DATA_DIR)

## Experiment identity
Change the owner, hypothesis, and config only. During the first backbone benchmark, all configs must share the same split, preprocessing, seed, image size, epoch budget, optimizer, and augmentation policy.

In [ ]:
OWNER = 'your_name'
EXPERIMENT_ID = 'E1a'
HYPOTHESIS = 'ResNet-18 transfer features will improve validation macro F1 over E0.'
CONFIG_PATH = REPO_ROOT / 'configs' / 'experiments' / 'resnet18_base.yaml'
print(OWNER, EXPERIMENT_ID, HYPOTHESIS, CONFIG_PATH, sep='\n')

## Preflight
The manifest files are created after dataset audit and duplicate-safe stratified 80/20 splitting. Training cannot start until those files and the shared trainer exist.

In [ ]:
required = [
    REPO_ROOT / 'data' / 'splits' / 'train.csv',
    REPO_ROOT / 'data' / 'splits' / 'val.csv',
    REPO_ROOT / 'data' / 'splits' / 'label_to_index.json',
    REPO_ROOT / 'src' / 'train.py',
    CONFIG_PATH,
]
missing = [str(path.relative_to(REPO_ROOT)) for path in required if not path.is_file()]
if missing:
    raise RuntimeError('Complete the shared audit/split/trainer gate first. Missing: ' + ', '.join(missing))
print('Preflight passed; shared dataset, split, trainer and model config exist.')

## Run the shared trainer
The trainer should save a run directory and print its `run_receipt.json` path. Use its receipt for the leaderboard; do not copy a rounded notebook score.

In [ ]:
import subprocess
command = [sys.executable, '-m', 'src.train', '--config', str(CONFIG_PATH)]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)

## Submit results
Record the full path to your run receipt and give it to the leaderboard owner. Discuss why validation macro F1, accuracy, and the per-class results changed relative to the baseline.

In [ ]:
import json
RECEIPT_PATH = REPO_ROOT / 'results' / 'runs' / 'SET_RUN_ID' / 'run_receipt.json'
if not RECEIPT_PATH.is_file():
    print('Set RECEIPT_PATH to the path printed by src.train after your run.')
else:
    receipt = json.loads(RECEIPT_PATH.read_text(encoding='utf-8'))
    print(json.dumps(receipt, ensure_ascii=False, indent=2))